# Module Extract — Collecte des données OpenWeather

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haja171106/donnee2-aqi/blob/feat/notebooks-analysis/notebooks/extract.ipynb)

Ce notebook analyse et documente le module `src/extract.py`.

**Rôle :** Interroger l'API OpenWeather Air Pollution pour 5 villes et sauvegarder les réponses brutes dans `data/raw/`.

**Deux modes :**
- `hourly` : collecte en temps réel (endpoint `air_pollution`)
- `backfill` : collecte historique (endpoint `air_pollution/history`)

**Fichiers produits :** `{ville}_{type}_{YYYYMMDDTHHMMSS}.json`

## 0. Configuration Colab

Cette cellule configure l'environnement que vous soyez dans Colab ou en local.

In [ ]:
import sys, os, json, time, requests
from datetime import datetime, timedelta, timezone
from pathlib import Path

# Détection Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    RAW_DIR = Path("/content/sample_data/raw")
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    # Télécharger des exemples de fichiers raw depuis GitHub
    import urllib.request
    sample_files = [
        "antananarivo_current_20260724T142544.json",
        "paris_current_20260724T142545.json",
        "beijing_history_20260724T190834.json",
    ]
    base = "https://raw.githubusercontent.com/haja171106/donnee2-aqi/feat/notebooks-analysis/data/raw"
    for fname in sample_files:
        urllib.request.urlretrieve(f"{base}/{fname}", RAW_DIR / fname)
    print(f"{len(sample_files)} fichiers sample téléchargés ✓")

    # Config depuis Colab secrets (optionnel)
    OPENWEATHER_API_KEY = userdata.get("OPENWEATHER_API_KEY", "")
    DATABASE_URL = userdata.get("DATABASE_URL", "")
else:
    from config import RAW_DIR, OPENWEATHER_API_KEY, DATABASE_URL
    print("Configuration locale ✓")

OW_CURRENT_URL = "https://api.openweathermap.org/data/2.5/air_pollution"
OW_HISTORY_URL = "https://api.openweathermap.org/data/2.5/air_pollution/history"

CITIES = [
    {"name": "Antananarivo", "country": "MG", "lat": -18.8792, "lon": 47.5079},
    {"name": "Paris", "country": "FR", "lat": 48.8566, "lon": 2.3522},
    {"name": "Nairobi", "country": "KE", "lat": -1.2921, "lon": 36.8219},
    {"name": "Mumbai", "country": "IN", "lat": 19.0760, "lon": 72.8777},
    {"name": "Beijing", "country": "CN", "lat": 39.9042, "lon": 116.4074},
]

print("Configuration terminée ✓")

---
## 1. Analyse des fonctions

### `_raw_path(city_name, kind, ts) -> Path`

Construit le chemin du fichier JSON à partir du nom de la ville, du type (`current` ou `history`) et de l'horodatage.

**Format :** `{ville}_{type}_{YYYYMMDDTHHMMSS}.json`

Exemple : `antananarivo_current_20260724T142544.json`

In [ ]:
def _raw_path(city_name: str, kind: str, ts: datetime) -> Path:
    stamp = ts.strftime("%Y%m%dT%H%M%S")
    fname = f"{city_name.lower()}_{kind}_{stamp}.json"
    return RAW_DIR / fname

exemple_ts = datetime(2026, 7, 24, 14, 25, 44, tzinfo=timezone.utc)
chemin = _raw_path("Antananarivo", "current", exemple_ts)
print(f"Chemin généré : {chemin.name}")

### `_save_raw(city_name, kind, payload)`

Sauvegarde la réponse JSON de l'API dans un fichier horodaté. Le fichier n'est **jamais modifié** après écriture — c'est la source de vérité du pipeline.

In [ ]:
def _save_raw(city_name: str, kind: str, payload: dict) -> None:
    path = _raw_path(city_name, kind, datetime.now(timezone.utc))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    print(f"Fichier sauvegardé : {path.name}")

print("Fonction _save_raw définie ✓")

### `fetch_current(lat, lon) -> dict`

Appelle l'endpoint `air_pollution` d'OpenWeather pour obtenir la qualité de l'air **actuelle** à des coordonnées données.

**Paramètres :**
- `lat`, `lon` : coordonnées de la ville

**Retourne :** la réponse JSON brute de l'API

**Documentation API :** [OpenWeather Air Pollution](https://openweathermap.org/api/air-pollution)

In [ ]:
def fetch_current(lat: float, lon: float) -> dict:
    resp = requests.get(
        OW_CURRENT_URL,
        params={"lat": lat, "lon": lon, "appid": OPENWEATHER_API_KEY},
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()

print("Fonction fetch_current définie ✓")
if not OPENWEATHER_API_KEY:
    print("⚠ clé API non configurée — définissez OPENWEATHER_API_KEY dans les secrets Colab pour exécuter")

### `fetch_history(lat, lon, start, end) -> dict`

Appelle l'endpoint `air_pollution/history` pour obtenir les données **historiques** entre deux timestamps UNIX.

**Paramètres :**
- `lat`, `lon` : coordonnées de la ville
- `start`, `end` : timestamps UNIX (secondes depuis epoch)

In [ ]:
def fetch_history(lat: float, lon: float, start: int, end: int) -> dict:
    resp = requests.get(
        OW_HISTORY_URL,
        params={"lat": lat, "lon": lon, "start": start, "end": end, "appid": OPENWEATHER_API_KEY},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()

print("Fonction fetch_history définie ✓")

### `run_hourly()`

Boucle sur les 5 villes et appelle `fetch_current` pour chacune. Ajoute les métadonnées de la ville (`_city_meta`) dans le payload, sauvegarde le fichier JSON, puis attend 1 seconde pour respecter le rate limit de l'API gratuite.

C'est cette fonction qui est exécutée **chaque heure** par GitHub Actions.

In [ ]:
def run_hourly() -> None:
    for city in CITIES:
        payload = fetch_current(city["lat"], city["lon"])
        payload["_city_meta"] = city
        _save_raw(city["name"], "current", payload)
        print(f"[hourly] {city['name']} OK")
        time.sleep(1)

print("Fonction run_hourly définie ✓")

### `run_backfill(months, chunk_days)`

Reconstruit l'historique en découpant la période en **tranches de 7 jours** (contrainte de l'API). Pour chaque ville, pour chaque tranche, appelle `fetch_history` et sauvegarde.

**Paramètres :**
- `months` : nombre de mois d'historique (défaut : 3)
- `chunk_days` : taille des tranches (défaut : 7)

In [ ]:
def run_backfill(months: int = 3, chunk_days: int = 7) -> None:
    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=30 * months)

    for city in CITIES:
        cursor = start_dt
        while cursor < end_dt:
            chunk_end = min(cursor + timedelta(days=chunk_days), end_dt)
            payload = fetch_history(
                city["lat"], city["lon"], int(cursor.timestamp()), int(chunk_end.timestamp())
            )
            payload["_city_meta"] = city
            _save_raw(city["name"], "history", payload)
            print(f"[backfill] {city['name']} {cursor.date()} -> {chunk_end.date()} OK")
            cursor = chunk_end
            time.sleep(1)

print("Fonction run_backfill définie ✓")

## 2. Analyse des données brutes produites

Regardons un exemple de fichier JSON produit par l'extraction.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.json"))
print(f"Nombre de fichiers raw disponibles : {len(raw_files)}")
for f in raw_files:
    print(f"  {f.name}")

In [ ]:
if raw_files:
    with open(raw_files[0]) as f:
        data = json.load(f)
    print(f"Fichier : {raw_files[0].name}")
    print(f"Clés racines : {list(data.keys())}")
    print(f"Ville : {data['_city_meta']['name']}")
    print(f"Pays : {data['_city_meta']['country']}")
    print(f"Coordonnées : {data['_city_meta']['lat']}, {data['_city_meta']['lon']}")
    print(f"Nombre de mesures : {len(data['list'])}")
    m = data['list'][0]
    print(f"Timestamp : {m['dt']} → {datetime.fromtimestamp(m['dt'], tz=timezone.utc)}")
    print(f"AQI : {m['main']['aqi']}")
    print(f"Polluants : {m['components']}")

## 3. Résumé

| Fonction | Rôle | Appelée par |
|---|---|---|
| `_raw_path()` | Construit le chemin du fichier JSON | `_save_raw()` |
| `_save_raw()` | Écrit le JSON sur le disque | `run_hourly()`, `run_backfill()` |
| `fetch_current()` | Appel API temps réel | `run_hourly()` |
| `fetch_history()` | Appel API historique | `run_backfill()` |
| `run_hourly()` | Boucle horaire sur 5 villes | GitHub Actions (etl.yml) |
| `run_backfill()` | Boucle historique par tranches de 7 jours | GitHub Actions (backfill.yml) |

**Points clés :**
- Les fichiers raw sont **immuables** : jamais modifiés après écriture
- Les métadonnées ville sont stockées dans `_city_meta` dans chaque fichier
- Rate limit : 1 seconde entre chaque appel API
- Quota gratuit : 1000 appels/jour → 5/heure = 120/jour utilisés